# Celery : un map-reduce réparti sur plusieurs workers

Cette démo exécute pour de vrai le map-reduce des diapositives sur Celery. L'application envoie des tâches à un broker ; des workers, des processus à part, les y prennent, les exécutent et rangent leurs résultats dans un backend, où l'application les lit. La démo compare ensuite la durée d'un même lot de tâches avec un worker, puis avec quatre workers derrière le même broker : c'est la répartition de charge de Celery. La cellule suivante installe Celery.

In [1]:
%pip install -q celery sqlalchemy

Note: you may need to restart the kernel to use updated packages.


## Le module des tâches

En production, le broker et le backend sont des services : Redis ou RabbitMQ, par exemple `broker="redis://localhost:6379/0"`. Pour que la démo n'ait aucun service à lancer, en local comme dans Colab, elle range les messages et les résultats dans deux bases SQLite, de simples fichiers : `sqla+sqlite://` est le transport SQLAlchemy de kombu (la bibliothèque de messagerie de Celery), `db+sqlite://` le backend SQLAlchemy de Celery. C'est lent, puisqu'un worker inactif ne relit la base qu'une fois par seconde, et ça ne sert qu'à une démonstration : seuls les processus de la machine qui voit ces fichiers peuvent en être les workers.

La tâche `carre` attend une seconde (`time.sleep`) avant de rendre son résultat, comme une tâche qui attend une réponse du réseau ; `somme` fera le reduce du map-reduce. Avec `worker_prefetch_multiplier = 1`, un worker ne réserve qu'une tâche d'avance, au lieu de quatre par processus par défaut, qu'un autre worker libre ne pourrait plus prendre. Avec `result_extended = True`, le backend garde aussi, avec chaque résultat, le nom du worker qui l'a calculé.

In [2]:
%%writefile taches.py
import time

from celery import Celery

app = Celery(
  "taches",
  broker="sqla+sqlite:///broker.sqlite",  # en production : "redis://localhost:6379/0"
  backend="db+sqlite:///resultats.sqlite",  # en production : "redis://localhost:6379/1"
)
app.conf.worker_prefetch_multiplier = 1  # une tâche réservée d'avance par processus
app.conf.result_extended = True  # nom du worker gardé avec chaque résultat


@app.task
def carre(x: int) -> int:
  time.sleep(1)  # une seconde d'attente : une requête réseau, un service lent…
  return x * x


@app.task
def somme(nombres: list[int]) -> int:
  return sum(nombres)

Writing taches.py


## Lancer et arrêter des workers

Un worker se lance d'habitude dans un terminal, par `celery -A taches worker`. Ici, chaque worker est un sous-processus du notebook, lancé avec le même interpréteur (`sys.executable`), et son journal va dans un fichier, `worker-1.log` pour le premier. Le gestionnaire de contexte `workers` vide d'abord la file (`app.control.purge`, pour les tâches d'une exécution interrompue), lance `nb` workers d'un processus chacun (`--concurrency=1`), attend que chacun écrive `ready.` dans son journal, puis les arrête à la sortie du bloc `with`, même si le code du bloc lève une exception.

In [3]:
import collections
import contextlib
import subprocess
import sys
import time
from pathlib import Path

from celery import chord, group

from taches import app, carre, somme


@contextlib.contextmanager
def workers(nb: int):
  app.control.purge()
  processus = []
  journaux = [Path(f"worker-{k}.log") for k in range(1, nb + 1)]
  try:
    for k, journal in enumerate(journaux, start=1):
      with journal.open("w") as sortie:
        processus.append(
          subprocess.Popen(
            [sys.executable, "-m", "celery", "-A", "taches", "worker"]
            + ["-n", f"worker{k}@demo", "--concurrency=1", "--loglevel=INFO"],
            stdout=sortie,
            stderr=subprocess.STDOUT,
          )
        )
    debut = time.perf_counter()
    while not all("ready." in journal.read_text() for journal in journaux):
      if time.perf_counter() - debut > 60:
        raise TimeoutError(f"{nb} workers pas prêts au bout de 60 s, voir worker-*.log")
      time.sleep(0.2)
    yield
  finally:
    for p in processus:
      p.terminate()  # arrêt à chaud : le worker finit d'abord sa tâche en cours
    for p in processus:
      p.wait(timeout=30)

## Un seul worker

`delay` envoie une tâche au broker et rend tout de suite un `AsyncResult`, dont `get` attend le résultat. Vient ensuite un `group` de 8 tâches : avec un seul worker d'un seul processus, elles passent l'une après l'autre, en 8 secondes au moins.

In [4]:
with workers(1):
  resultat = carre.delay(4)
  print(resultat.get(timeout=10))

  debut = time.perf_counter()
  carres = group(carre.s(i) for i in range(8))()
  print(carres.get(timeout=30))
  print(f"1 worker : {time.perf_counter() - debut:.1f} s")

16
[0, 1, 4, 9, 16, 25, 36, 49]
1 worker : 9.6 s


## Quatre workers derrière le même broker

Les mêmes 8 tâches, cette fois avec quatre workers qui lisent le même broker : chacun prend la tâche suivante dès qu'il est libre, sans que l'application sache combien de workers tournent. Pour monter en charge, on lance donc des workers de plus, sur la même machine ou sur d'autres machines qui voient le broker, sans rien changer à l'application. L'attribut `worker` de chaque résultat (grâce à `result_extended`) montre la répartition.

In [5]:
with workers(4):
  debut = time.perf_counter()
  carres = group(carre.s(i) for i in range(8))()
  print(carres.get(timeout=30))
  print(f"4 workers : {time.perf_counter() - debut:.1f} s")

  repartition = collections.Counter(resultat.worker for resultat in carres.results)
  for nom, nb_taches in sorted(repartition.items()):
    print(nom, nb_taches)

[0, 1, 4, 9, 16, 25, 36, 49]
4 workers : 3.2 s
worker1@demo 2
worker2@demo 2
worker3@demo 2
worker4@demo 2


## Le map-reduce : `chord`

Le `chord` des diapositives : les 8 carrés de l'en-tête (le map) s'exécutent en parallèle sur les quatre workers, puis le rappel `somme` (le reduce) reçoit la liste de leurs résultats, 0 + 1 + 4 + … + 49 = 140. Avec ce backend, une tâche de Celery (`chord_unlock`) vérifie chaque seconde si tout l'en-tête a fini, d'où environ une seconde de plus qu'avec le `group` ; Redis prévient le rappel dès le dernier résultat.

In [6]:
with workers(4):
  debut = time.perf_counter()
  total = chord(carre.s(i) for i in range(8))(somme.s())
  print(total.get(timeout=30))
  print(f"chord sur 4 workers : {time.perf_counter() - debut:.1f} s")

140
chord sur 4 workers : 4.3 s


## En production

Le même code tourne avec un broker Redis ou RabbitMQ : seules les URL de `Celery(...)` changent. Sur une machine, un seul worker de plusieurs processus remplace les quatre workers de cette démo (`celery -A taches worker --concurrency=4`), ou ajuste leur nombre à la longueur de la file (`--autoscale=8,2` : entre 2 et 8 processus). Sur plusieurs machines, chacune lance ses workers vers le même broker. Avec Redis ou RabbitMQ, on supervise aussi les workers : `celery -A taches inspect active` liste les tâches en cours de chacun, et Flower (`pip install flower`, puis `celery -A taches flower`) les montre dans un tableau de bord web. Le broker SQLite de cette démo ne transmet pas ces commandes de supervision, d'où la lecture des journaux pour savoir quand les workers sont prêts.